# ЛР-4. Машины опорных векторов (SVM)

Датасет: MOABB BNCI2014009 (P300 BCI).  
Алгоритм: Linear SVM (Pegasos — стохастический субградиентный спуск).  
Задача: классификация Target (P300) vs NonTarget.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from mlcore.data.datasets import load_lr4_dataset
from mlcore.preprocessing.scaling import standardize
from mlcore.preprocessing.splitting import train_test_split
from mlcore.metrics import accuracy, precision, recall, f1_score, confusion_matrix, roc_curve, auc_score
from mlcore.plotting.evaluation import plot_confusion_matrix, plot_roc_curve
from mlcore.utils.timing import timer

ROOT = Path.cwd()
for p in [ROOT, ROOT.parent, ROOT.parent.parent, ROOT.parent.parent.parent]:
    if (p / 'mlcore').exists():
        ROOT = p
        break
ASSETS = ROOT / 'labs/lr-4/assets'
ASSETS.mkdir(parents=True, exist_ok=True)

## 1. Загрузка и анализ данных

MOABB BNCI2014009: 10 субъектов, 16 EEG-каналов, 256 Гц.  
Используем MNE для извлечения эпох (это предобработка данных, не ML-моделирование).

In [ ]:
import mne
mne.set_log_level('WARNING')

dataset = load_lr4_dataset()
print(f'Dataset: {dataset.code}')
print(f'Subjects: {dataset.subject_list}')

In [ ]:
# Extract epochs from subjects 1-3 (balance speed vs data size)
SUBJECTS = [1, 2, 3]
all_epochs_data = []
all_labels = []

for subj in SUBJECTS:
    data = dataset.get_data(subjects=[subj])
    for session_name, session_data in data[subj].items():
        for run_name, raw in session_data.items():
            events, event_id = mne.events_from_annotations(raw)
            # Map event IDs to Target(1) / NonTarget(0)
            target_ids = [v for k, v in event_id.items() if 'Target' in k or 'target' in k.lower()]
            nontarget_ids = [v for k, v in event_id.items() if 'NonTarget' in k or 'nontarget' in k.lower()]
            if not target_ids or not nontarget_ids:
                # Try numeric mapping
                target_ids = [v for k, v in event_id.items() if k.strip() == '2']
                nontarget_ids = [v for k, v in event_id.items() if k.strip() == '1']
            if not target_ids or not nontarget_ids:
                continue

            selected_events = events[np.isin(events[:, 2], target_ids + nontarget_ids)]
            epochs = mne.Epochs(raw, selected_events, tmin=0, tmax=0.8,
                                baseline=None, preload=True, verbose=False)
            epoch_data = epochs.get_data()  # (n_epochs, n_channels, n_times)
            labels = np.array([1 if e in target_ids else 0 for e in selected_events[:len(epoch_data), 2]])
            all_epochs_data.append(epoch_data)
            all_labels.append(labels)

all_epochs = np.concatenate(all_epochs_data)
all_labels = np.concatenate(all_labels)
print(f'Total epochs: {len(all_epochs)}, shape: {all_epochs.shape}')
print(f'Target: {(all_labels == 1).sum()}, NonTarget: {(all_labels == 0).sum()}')

## 2. Feature Engineering

Для каждого канала вычисляем среднюю амплитуду в 5 временных окнах (0-160, 160-320, 320-480, 480-640, 640-800 мс).  
16 каналов × 5 окон = 80 признаков.

In [ ]:
def extract_features(epochs_data: np.ndarray, n_windows: int = 5) -> np.ndarray:
    """Time-windowed channel means."""
    n_epochs, n_channels, n_times = epochs_data.shape
    window_size = n_times // n_windows
    features = np.zeros((n_epochs, n_channels * n_windows))
    for w in range(n_windows):
        start = w * window_size
        end = start + window_size if w < n_windows - 1 else n_times
        features[:, w * n_channels:(w + 1) * n_channels] = epochs_data[:, :, start:end].mean(axis=2)
    return features

X_all = extract_features(all_epochs)
y_all = all_labels
print(f'Feature matrix: {X_all.shape}')

In [ ]:
# Undersample NonTarget to 2:1 ratio
target_idx = np.where(y_all == 1)[0]
nontarget_idx = np.where(y_all == 0)[0]
rng = np.random.default_rng(42)
n_keep = min(len(nontarget_idx), 2 * len(target_idx))
nontarget_sample = rng.choice(nontarget_idx, size=n_keep, replace=False)
keep_idx = np.sort(np.concatenate([target_idx, nontarget_sample]))

X_bal = X_all[keep_idx]
y_bal = y_all[keep_idx]
print(f'After undersampling: {len(X_bal)} (Target={sum(y_bal==1)}, NonTarget={sum(y_bal==0)})')

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
)
X_train, mean, std = standardize(X_train)
X_test, _, _ = standardize(X_test, mean=mean, std=std)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 3. Linear SVM (Pegasos)

Стохастический субградиентный спуск на hinge loss:  
`L = (lambda/2) ||w||^2 + max(0, 1 - y*(w@x + b))`

Обоснование линейного ядра: ERP-признаки (80 измерений) — достаточно высокая размерность для линейной разделимости.

In [ ]:
class LinearSVM:
    """Linear SVM trained via Pegasos (sub-gradient descent)."""

    def __init__(self, C=1.0, max_iter=2000, random_state=42):
        self.C = C
        self.lam = 1.0 / C
        self.max_iter = max_iter
        self.rng = np.random.default_rng(random_state)
        self.w = None
        self.b = 0.0
        self.loss_history = []

    def fit(self, X, y):
        """y must be in {-1, +1}."""
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        self.loss_history = []

        for t in range(1, self.max_iter + 1):
            eta = 1.0 / (self.lam * t)
            i = self.rng.integers(n)
            margin = y[i] * (X[i] @ self.w + self.b)

            if margin < 1:
                self.w = (1 - eta * self.lam) * self.w + eta * y[i] * X[i]
                self.b += eta * y[i]
            else:
                self.w = (1 - eta * self.lam) * self.w

            if t % 100 == 0:
                scores = X @ self.w + self.b
                hinge = np.maximum(0, 1 - y * scores).mean()
                reg = 0.5 * self.lam * np.dot(self.w, self.w)
                self.loss_history.append(hinge + reg)

        return self

    def decision_function(self, X):
        return X @ self.w + self.b

    def predict(self, X):
        return np.sign(self.decision_function(X)).astype(int)

print('LinearSVM defined.')

In [ ]:
# Convert labels to {-1, +1}
y_train_svm = 2 * y_train - 1
y_test_svm = 2 * y_test - 1

svm = LinearSVM(C=1.0, max_iter=3000, random_state=42)
with timer('SVM training'):
    svm.fit(X_train, y_train_svm)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(svm.loss_history)
ax.set_xlabel('Iteration (x100)')
ax.set_ylabel('Hinge + Regularization')
ax.set_title('SVM Training Loss')
fig.tight_layout()
fig.savefig(ASSETS / 'training_loss.png', dpi=150)
plt.close(fig)
print('Saved training_loss.png')

## 4. Оценка качества

In [ ]:
y_pred_svm = svm.predict(X_test)
y_pred_01 = ((y_pred_svm + 1) // 2).astype(int)  # back to {0, 1}

metrics = {
    'Accuracy': accuracy(y_test, y_pred_01),
    'Precision': precision(y_test, y_pred_01, average='macro'),
    'Recall': recall(y_test, y_pred_01, average='macro'),
    'F1-score': f1_score(y_test, y_pred_01, average='macro'),
}
for k, v in metrics.items():
    print(f'{k}: {v:.4f}')

In [ ]:
cm = confusion_matrix(y_test, y_pred_01)
fig = plot_confusion_matrix(cm, class_names=['NonTarget', 'Target'])
fig.savefig(ASSETS / 'confusion_matrix.png', dpi=150)
plt.close(fig)
print('Saved confusion_matrix.png')

In [ ]:
# ROC: use decision function scores as proxy for probabilities
scores = svm.decision_function(X_test)
# Platt-like scaling via sigmoid
proba = 1 / (1 + np.exp(-scores))

fpr, tpr, _ = roc_curve(y_test, proba)
auc = auc_score(fpr, tpr)
print(f'AUC: {auc:.4f}')

fig = plot_roc_curve(fpr, tpr, auc=auc)
fig.savefig(ASSETS / 'roc_curve.png', dpi=150)
plt.close(fig)
print('Saved roc_curve.png')

## 5. Выводы

1. Linear SVM (Pegasos) реализован с нуля на numpy.
2. Линейное ядро обосновано высокой размерностью ERP-признаков.
3. Undersampling NonTarget до 2:1 критически важен из-за дисбаланса 1:5.
4. P300 detection — сложная задача; accuracy 60-75% является нормой для single-trial.